# ScoreLine pipeline walkthrough
Runs each stage of the skeleton pipeline. Run from `ai/training`.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root
from ai.training import preprocessing as pp, train as tr
from ai.training.pipeline import CONFIG

## 1. Objective & metrics

In [ ]:
CONFIG

## 2. Acquire / parse

In [ ]:
# Real data (needs API key / backend running):
# import asyncio
# from backend.app.services.football_api import FootballAPIClient
# matches = asyncio.run(FootballAPIClient(...).get_matches(...))
results = ["H", "D", "A", "H", "H"] * 4
matches = [
    {"id": i, "home_team": "X", "away_team": "Y", "date": f"2026-01-{i + 1:02d}T15:00:00Z",
     "status": "FINISHED", "home_score": {"H": 2, "D": 1, "A": 0}[r], "away_score": 1 if r != "A" else 2}
    for i, r in enumerate(results)
]
df = pp.load_api_matches(matches)
df.head()

## 3. Inspect

In [ ]:
pp.inspect(df)

## 4. Clean

In [ ]:
df = pp.clean(df)
len(df)

## 5. Label + time split (before exploring)

In [ ]:
df = pp.label_result(df)
train_df, test_df = pp.time_split(df, CONFIG['test_frac'])
len(train_df), len(test_df)

## 6. Explore (train only)

In [ ]:
# TODO: richer EDA
train_df['result'].value_counts()

## 7. Feature engineering

In [ ]:
X_train, X_test = tr.build_features(train_df), tr.build_features(test_df)
y_train, y_test = train_df[CONFIG['target']], test_df[CONFIG['target']]

## 7b. Model selection (TimeSeriesSplit CV on train, mean log loss)

In [ ]:
best_name, model, scores = tr.select_model(X_train, y_train, n_splits=CONFIG['cv_splits'])
print(best_name)
scores

## 8. Final evaluate (selected model on held-out test)

In [ ]:
tr.evaluate(model, X_test, y_test)